# Simple Microgrid Kaggle Notebook

用于在 Kaggle 上运行 `microgrid_sim` 的 DRL、Oracle 和 heuristic-lite 对照，并导出论文可用结果工件。

默认主链路：
- `scripts/analysis/short_cross_fidelity_probe.py`
- 可选：`scripts/analysis/full_year_oracle_compare.py`
- 可选：`scripts/analysis/full_year_heuristic_lite_compare.py`

默认已经切到严肃长跑配置：`IEEE33 + SAC + 300k + 365d held-out eval`。

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Has /kaggle/input:', Path('/kaggle/input').exists())
print('Has /kaggle/working:', Path('/kaggle/working').exists())
print('CPU count:', os.cpu_count())


## 0) 参数区

只改这一格。Notebook 后续所有命令都会从这里读取配置。

In [ ]:
from dataclasses import dataclass


@dataclass
class RunCfg:
    # ============================
    # 0) Kaggle 输入/输出路径约定
    # ============================

    # Kaggle 代码数据集目录（/kaggle/input 下的只读目录）。
    # 推荐填“仓库根目录”，而不是 `.../src/microgrid_sim` 这种子目录。
    # 原因：notebook 后续需要找到：
    # - 仓库根的 `pyproject.toml`（editable install）
    # - `scripts/`（分析脚本入口）
    # - `data/`（数据布局约定，尤其是 processed）
    # - `src/`（包源码）
    # 备注：当前 notebook 也支持你误填到子目录时自动向上寻找 `pyproject.toml`，但仍建议按根目录填最稳。
    kaggle_codeset_dir: str = '/kaggle/input/datasets/tttwwwjjj/microgrid-code'

    # Kaggle 数据数据集目录（/kaggle/input 下的只读目录）。
    # 建议直接指向 `.../microgrid-data/data/processed`（即 processed 的父目录下包含 `network_15min/`）。
    # notebook 会把该目录复制到 `work_root/data/processed`，从而兼容仓库的默认数据解析逻辑。
    # 这与 README 的 network-first 数据布局一致：
    # - `data/processed/network_15min/<case>/load.csv,pv.csv,price.csv`
    kaggle_data_processed_dir: str = '/kaggle/input/datasets/tttwwwjjj/microgrid-data/data/processed'

    # 可选：上一次运行导出的 Kaggle 输出目录，用于把旧的 `results/` 合并进当前工作目录。
    # 用途：断点续跑/复用已有工件（例如 tensorboard 或历史 summary）。
    # 不需要续跑时保持空字符串即可。
    kaggle_prev_output_dir: str = ''

    # Kaggle 可写工作目录（/kaggle/working）。
    # 代码会被复制到这里，然后执行 `pip install -e .`。
    # 这样可以避免在只读的 `/kaggle/input` 目录上直接运行导致权限问题。
    work_root: str = '/kaggle/working/microgrid_sim'

    # 是否把 `/kaggle/input` 中的代码数据集复制到 `work_root`。
    # 默认建议 True：
    # - 可避免只读目录写入失败
    # - 同时让后续的 `pip install -e` 安装路径稳定
    sync_code_to_workdir: bool = True

    # 最小依赖集合：显式安装 RL（SB3）、潮流（pandapower）、结果分析（pandas/matplotlib）等必需包。
    # 若你在 Kaggle 环境里已经预装了其中一部分，这里重复安装通常也不会影响正确性。
    # 注意：这里采用 pip 安装，不走本仓库的 uv lock；Kaggle 复现优先保证“能跑”。
    install_packages: tuple[str, ...] = (
        'numpy',
        'pandas',
        'scipy',
        'matplotlib',
        'gymnasium',
        'stable-baselines3',
        'sb3-contrib',
        'pandapower',
        'tensorboard',
        'tqdm',
    )

    # ============================
    # 1) 实验命名与主实验场景
    # ============================

    # 实验名会直接用于：
    # - 输出目录：`results/<experiment_name>/`
    # - 压缩包命名：`results/<experiment_name>.zip`
    # 建议把关键协议写进名字，避免后续聚合/制表时混淆：
    # - case / agent / train_steps
    # - train_year / eval_year
    # - eval_days 是否是全年（365d）
    # 参考：CLI.md 与 kaggle.md 的推荐命名方式。
    experiment_name: str = 'kaggle_ieee33_sac_300k_train2023_eval2024_365d'

    # 主实验电网（case）。
    # 当前仓库的网络级主线环境是 `NetworkMicrogridEnv`，并内置 canonical profiles：
    # - `ieee33`（推荐作为 DRL 主实验台，网络压力更适合做 reviewer-facing 论证）
    # - `cigre`（增强后的次实验台，更偏“价值存在但控制恢复更难”的对照）
    # 详见 docs/project_document/04_network_and_battery_modeling.md 与 13_drl_readiness_and_365d_protocol.md。
    case: str = 'ieee33'

    # 工况（regime）。
    # 该字段会进入 `NetworkProfiles` 的缩放逻辑（src/microgrid_sim/data/network_profiles.py）。
    # 当前合法选项为：
    # - `base`：不额外施加 stress，作为名义工况/对照。
    # - `high_load`：整体负荷放大，并额外增强傍晚时段负荷（更强晚高峰压力）。
    # - `high_pv`：整体 PV 放大，并额外增强中午 PV（用于测试中午反向潮流/电压抬升敏感性）。
    # - `network_stress`：同时提高负荷、降低 PV，并放大价格波动；在 IEEE33 上还会额外强化 shoulder/late-evening。
    #   这是当前论文/审稿导向的主场景：更容易放大网络约束与储能调度价值。
    # - `tight_soc`：让电池初始 SOC 被重置到较低且较窄的区间（当前实现为 0.15~0.30），并轻微放大负荷。
    #   用途：故意让“可调度空间”变小，测试策略在库存受限时是否仍能保持闭环与可行性。
    #
    # 实践建议：
    # - 做主结果：优先 `network_stress`（与 CLI.md / kaggle.md 推荐一致）。
    # - 做机理对照：用 `base/high_load/high_pv` 分别隔离压力来源（负荷、PV、价格波动）。
    # - 做鲁棒性/边界测试：用 `tight_soc` 看策略是否依赖“初始 SOC 运气”。
    regime: str = 'network_stress'

    # 奖励配置（reward_profile）。
    # 该字段会在 config 的 `__post_init__` 中选择不同的 reward 系数组合（见 src/microgrid_sim/cases.py）：
    # - `network`：更贴近“网络运行”导向的默认配置（CLI 的 smoke test 默认就是它）。
    #   特点：不同 case（IEEE33/CIGRE）可能有各自的默认 reward 系数，更多用于环境 sanity check。
    # - `paper_aligned`：更接近参考论文的系数标定。
    #   特点：会把 `battery_throughput/loss/stress` 这三类额外惩罚关闭，减少 reward 项，便于做“对齐参考设定”的对照。
    # - `paper_balanced`：当前最推荐的“论文可用 + 训练更稳”的折中。
    #   特点：
    #   - cost 仍是主导（w_cost=1.0）
    #   - 网络约束项（电压/线路/变压器）权重较高，避免策略只做能量套利不管网
    #   - 增加适度 shaping（w_band/w_edge）来降低 SOC 边界吸引子
    #   - 启用 `battery_throughput/loss/stress` 的轻量惩罚，抑制过度充放电与高损耗行为
    #   - 同时会设置 terminal SOC closure（target/tolerance/penalty），让“全年/长窗”策略更难靠末端把电池打空取巧
    #
    # 实践建议：
    # - Smoke（只验证链路）：`network`
    # - Serious 年度长跑/论文主结果：`paper_balanced`
    # - 若你明确要复刻某个参考系数表：再切 `paper_aligned`
    reward_profile: str = 'paper_balanced'

    # DRL 算法（agent）。
    # 实际可选范围由 `src/microgrid_sim/rl_utils.py` 定义，目前支持：
    # - `sac`, `ppo`, `td3`, `ddpg`, `d4pg`, `dqn`, `tqc`, `trpo`
    # 其中：
    # - SAC/TD3/DDPG/TQC 等为 off-policy（需要 replay buffer），一般更适合连续动作。
    # - PPO/TRPO 为 on-policy，更新更稳定但样本效率可能更低。
    # - DQN 是离散动作算法：只有当环境 action space 是 Discrete 时才适用。
    #
    # 当前默认用 SAC：
    # - 经验上它在本项目的早期训练阶段更不容易立刻掉进低 SOC 边界吸引子
    # - 且对连续动作更自然
    # 若你要做算法对照：建议至少补一条 PPO（保持相同 protocol），再视情况加 TD3/TQC。
    agent: str = 'sac'

    # ============================
    # 2) Battery fidelity：训练/评估电池模型
    # ============================

    # 训练电池模型（train_model）与评估电池模型（test_model）。
    # 这是本项目“跨保真度/跨年度”主线的核心对照轴之一。
    # 注意：这里的字符串会直接传给 `short_cross_fidelity_probe.py` 的 CLI 参数：
    # - `--train-models <train_model>`
    # - `--test-models <test_model>`
    # 当前常用取值：
    # - `simple`：低保真（SimpleBattery）
    # - `thevenin_loss_only`：loss-aware Thevenin（仍是 TheveninBattery 的参数化模式）
    # - `thevenin`：full-physics Thevenin
    # - 以及 mixed-fidelity 训练写法（例如 `simple+thevenin`）
    # 更完整的层次说明见 docs/project_document/04_network_and_battery_modeling.md。
    #
    # 默认 `simple -> simple`：
    # - 最稳的主结果候选线
    # - 最容易先拿到“收敛证据/形态稳定证据”
    # 若要做跨保真部署失配（train-test mismatch）：
    # - 可把 `test_model` 改成 `thevenin`
    # - 或把 `train_model` 改成 `simple+thevenin` 做两阶段/混合保真训练
    train_model: str = 'simple'
    test_model: str = 'simple'

    # ============================
    # 3) 训练/评估协议（Year-split + 365d full-horizon）
    # ============================

    # 训练步数（SB3 的 time steps）。
    # Kaggle 分档建议（来自 kaggle.md）：
    # - Smoke：`5k`（只验证安装、数据、日志、输出工件链路）
    # - Scout：`10k ~ 20k`（摸参数、比较算法/电网组合的策略形态）
    # - Serious：`100k+`，严肃比较建议 `200k ~ 300k`
    train_steps: int = 300000

    # 环境基准长度（天）。
    # `days=365` 的含义是：环境/数据以全年长度定义。
    # 重要：它本身并不自动等于“全年评估”。
    # 是否真正全年评估，要看 `eval_days` 与 `eval_full_horizon`。
    # 参考：README.md 与 CLI.md 的判读提醒。
    days: int = 365

    # 年度切分协议：训练年 / 评估年。
    # 默认使用最推荐的 held-out generalization 设置：
    # - train_year = 2023
    # - eval_year = 2024
    # 该协议的必要性与常见误区详见 docs/project_document/11_year_split_generalization_assessment.md。
    train_year: int = 2023
    eval_year: int = 2024

    # 训练时每个 episode 的采样窗口长度（天）。
    # 30 天是当前推荐折中：
    # - 足够长：让策略看到更明显的库存闭环/跨日套利结构
    # - 又不过长：避免 episode 太长导致训练更新太慢、信用分配更难
    train_episode_days: int = 30

    # 正式评估窗口长度（天）。
    # serious 年度证据建议直接 `365`。
    # 若只是试运行/快速 sanity check，可改成 `7` 或 `30`。
    eval_days: int = 365

    # 是否启用“全窗口/全 horizon 评估”。
    # 若为 True：忽略 eval_steps，直接滚完整个 `eval_days`。
    # 这是全年结论的硬性要求之一（否则很容易出现：days=365 但 eval 只有 96 steps=1 day 的误读）。
    # 参考：kaggle.md / CLI.md / README.md 对 `--eval-full-horizon` 的强调。
    eval_full_horizon: bool = True

    # 随机种子。
    # 论文级结论建议多 seed；但 Kaggle 单次长跑通常先从 seed42 起步。
    seed: int = 42

    # 设备选择。
    # `auto` 会让 SB3 尝试使用 GPU（若 Kaggle 环境可用），否则会回退到 CPU。
    # 注意：潮流计算（pandapower）主要仍在 CPU 上，因此 GPU 并不一定带来整体同等比例加速。
    device: str = 'auto'

    # ============================
    # 4) 连续动作稳定化正则（强烈建议保持开启）
    # ============================

    # 连续动作正则三件套（来自 kaggle.md 的“固定正则配置”）：
    # - smoothing：抑制 action 抖动
    # - max_delta：限制步间突变（防止高频剧烈充放电切换）
    # - rate_penalty：对高频来回切换施加惩罚
    # 这套配置的目的主要是：
    # - 减少“边界僵尸策略”（长期粘在 SOC 下界/上界）
    # - 减少不可行动作刷屏
    action_smoothing_coef: float = 0.5
    action_max_delta: float = 0.1
    action_rate_penalty: float = 0.05

    # ============================
    # 5) 电池可行域感知（feasibility-aware）与动作对称化
    # ============================

    # 是否启用 SOC 可行域感知。
    # 启用后：当动作请求与当前 SOC 状态矛盾时（例如 SOC 已接近下界还想放电），
    # 环境会更明确地暴露“不可行差距（infeasible gap）”，并可施加额外惩罚。
    # 这通常能显著改善策略学到的库存闭环形态。
    battery_feasibility_aware: bool = True

    # 对不可行动作施加的负奖励系数。
    # 数值越负，越会惩罚：
    # - “想放电但其实已经没电”
    # - “想充电但其实已经满了”
    # 常用经验：先保持 -1.0，若仍出现大量 infeasible action dwell，可逐步加大惩罚幅度。
    battery_infeasible_penalty: float = -1.0

    # 是否对 battery action 做对称化缩放。
    # 直觉：让正向/负向动作在“可用充电/放电空间”上更对称，有助于减少单边偏置。
    symmetric_battery_action: bool = True

    # ============================
    # 6) 可选：全年上界/快线 baseline
    # ============================

    # 是否顺带跑全年 Oracle（MILP/LP perfect-foresight 上界）。
    # 作用：回答“该 case+regime 下电池理论上有没有价值”。
    # 对论文写作/benchmark 叙述非常关键。
    # 参考：CLI.md 与 docs/project_document/13_drl_readiness_and_365d_protocol.md。
    run_oracle: bool = True

    # 是否顺带跑全年 heuristic-lite。
    # 这是一个全年“快速次优对照”，用于在不做重型 GA/DRL 的情况下快速估计：
    # - 与 Oracle 的差距
    # - 简单规则能恢复多少价值
    # 默认关闭：因为它会增加运行时长与输出工件体积；需要做 gap 分析时再打开。
    run_heuristic_lite: bool = False


cfg = RunCfg()
cfg

## 1) 同步代码到 `/kaggle/working`

In [ ]:
from pathlib import Path
import shutil


def find_codeset(preferred: str) -> Path:
    preferred_path = Path(preferred)
    if preferred_path.exists():
        for candidate in [preferred_path, *preferred_path.parents]:
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
                return candidate
    candidates = []
    root = Path('/kaggle/input')
    for path in root.rglob('*'):
        if path.is_dir() and (path / 'pyproject.toml').exists() and (path / 'src').exists():
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError('No Kaggle input dataset containing pyproject.toml and src/ was found.')
    return candidates[0]


def find_processed_data_dir(preferred: str) -> Path | None:
    if not preferred:
        return None
    preferred_path = Path(preferred)
    if preferred_path.exists():
        if (preferred_path / 'network_15min').exists():
            return preferred_path
        if preferred_path.name == 'network_15min':
            return preferred_path.parent
    root = Path('/kaggle/input')
    for path in root.rglob('processed'):
        if (path / 'network_15min').exists():
            return path
    return None


source_repo = find_codeset(cfg.kaggle_codeset_dir)
processed_data_dir = find_processed_data_dir(cfg.kaggle_data_processed_dir)
work_root = Path(cfg.work_root)
if cfg.sync_code_to_workdir:
    if work_root.exists():
        shutil.rmtree(work_root)
    shutil.copytree(source_repo, work_root)

if processed_data_dir is not None:
    target_processed_dir = work_root / 'data' / 'processed'
    target_processed_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(processed_data_dir, target_processed_dir, dirs_exist_ok=True)

if cfg.kaggle_prev_output_dir and Path(cfg.kaggle_prev_output_dir).exists():
    prev_dir = Path(cfg.kaggle_prev_output_dir)
    if (prev_dir / 'results').exists():
        shutil.copytree(prev_dir / 'results', work_root / 'results', dirs_exist_ok=True)

print('source_repo =', source_repo)
print('processed_data_dir =', processed_data_dir)
print('work_root =', work_root)


## 2) 安装依赖并安装仓库

In [ ]:
import subprocess
import sys


subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *cfg.install_packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(work_root)], check=True)
print('editable install done')


## 3) 运行 DRL 主实验

In [ ]:
from pathlib import Path
import subprocess
import sys


output_dir = work_root / 'results' / cfg.experiment_name
output_dir.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(work_root / 'scripts' / 'analysis' / 'short_cross_fidelity_probe.py'),
    '--cases', cfg.case,
    '--regimes', cfg.regime,
    '--reward-profile', cfg.reward_profile,
    '--agent', cfg.agent,
    '--train-models', cfg.train_model,
    '--test-models', cfg.test_model,
    '--train-steps', str(cfg.train_steps),
    '--eval-steps', '0',
    '--days', str(cfg.days),
    '--train-year', str(cfg.train_year),
    '--train-episode-days', str(cfg.train_episode_days),
    '--train-random-start-within-year',
    '--eval-year', str(cfg.eval_year),
    '--eval-days', str(cfg.eval_days),
    '--seed', str(cfg.seed),
    '--device', cfg.device,
    '--action-smoothing-coef', str(cfg.action_smoothing_coef),
    '--action-max-delta', str(cfg.action_max_delta),
    '--action-rate-penalty', str(cfg.action_rate_penalty),
    '--output-dir', str(output_dir),
]

if cfg.eval_full_horizon:
    cmd.append('--eval-full-horizon')
if cfg.battery_feasibility_aware:
    cmd.append('--battery-feasibility-aware')
    cmd.extend(['--battery-infeasible-penalty', str(cfg.battery_infeasible_penalty)])
if cfg.symmetric_battery_action:
    cmd.append('--symmetric-battery-action')

print('Running command:')
print(' '.join(cmd))
completed = subprocess.run(cmd, cwd=work_root, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr.strip():
    print('STDERR:')
    print(completed.stderr)


## 4) 读取结果摘要

In [ ]:
import pandas as pd


summary_csv = output_dir / 'summary.csv'
summary_df = pd.read_csv(summary_csv)
display_cols = [
    'case',
    'regime',
    'agent',
    'train_model',
    'test_model',
    'train_steps',
    'steps',
    'final_cumulative_cost',
    'final_cumulative_objective_cost',
    'final_soc',
    'total_terminal_soc_penalty',
    'soc_lower_dwell_fraction',
    'soc_upper_dwell_fraction',
    'infeasible_action_dwell_fraction',
    'mean_battery_action_infeasible_gap',
]
summary_df[display_cols]


## 5) 可选：运行全年 Oracle

In [ ]:
if cfg.run_oracle:
    oracle_dir = work_root / 'results' / f'{cfg.experiment_name}_oracle'
    oracle_cmd = [
        sys.executable,
        str(work_root / 'scripts' / 'analysis' / 'full_year_oracle_compare.py'),
        '--cases', cfg.case,
        '--regimes', cfg.regime,
        '--reward-profile', cfg.reward_profile,
        '--battery-model', 'simple',
        '--days', '365',
        '--seed', str(cfg.seed),
        '--efficiency-model', 'realistic',
        '--output-dir', str(oracle_dir),
    ]
    print(' '.join(oracle_cmd))
    oracle_completed = subprocess.run(oracle_cmd, cwd=work_root, text=True, capture_output=True, check=True)
    print(oracle_completed.stdout)
else:
    print('skip oracle')


## 6) 可选：运行全年 heuristic-lite

In [ ]:
if cfg.run_heuristic_lite:
    heuristic_dir = work_root / 'results' / f'{cfg.experiment_name}_heuristic_lite'
    oracle_summary_csv = work_root / 'results' / f'{cfg.experiment_name}_oracle' / 'protocol_summary.csv'
    heuristic_cmd = [
        sys.executable,
        str(work_root / 'scripts' / 'analysis' / 'full_year_heuristic_lite_compare.py'),
        '--cases', cfg.case,
        '--regimes', cfg.regime,
        '--reward-profile', cfg.reward_profile,
        '--battery-model', 'simple',
        '--days', '365',
        '--seed', str(cfg.seed),
        '--baselines', 'none,heuristic_blended,heuristic_selector_lite',
        '--selector-window-days', '7',
        '--selector-stride-days', '1',
        '--evaluation-mode', 'surrogate',
        '--oracle-summary-csv', str(oracle_summary_csv),
        '--output-dir', str(heuristic_dir),
    ]
    print(' '.join(heuristic_cmd))
    heuristic_completed = subprocess.run(heuristic_cmd, cwd=work_root, text=True, capture_output=True, check=True)
    print(heuristic_completed.stdout)
else:
    print('skip heuristic-lite')


## 7) 打包输出

In [ ]:
import shutil


archive_base = work_root / 'results' / cfg.experiment_name
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=output_dir)
print('archive_path =', archive_path)
